# 03b — Training ResNet50

**Fase 2** — Training ResNet50 x 5 seed x 2 stage (freeze -> fine-tune)

**Referensi:** Proposal 3.3.6, 3.3.7

---

**Konfigurasi:**
- Stage 1: Freeze base model, LR=1e-3, 10 epoch
- Stage 2: Unfreeze 20 layer terakhir, LR=1e-4, 10 epoch
- Callbacks: EarlyStopping(p=5), ReduceLROnPlateau(f=0.5,p=5), ModelCheckpoint
- 5 seed: [42, 123, 2024, 7, 99]

**Output:**
- Checkpoint: `models/resnet50_seed{n}.keras`
- History JSON: `results/history_resnet50_seed{n}.json`

## Langkah 1 — Setup Environment (Clone Repo)

In [ ]:
# ================================================================
# SETUP: Clone repo dari GitHub LFS
# ================================================================
# Kalau repo sudah di-clone, comment/block cell ini
import subprocess
result = subprocess.run(['ls', str(REPO / 'src')], capture_output=True)
if result.returncode == 0 and (REPO / 'src' / 'model_builder.py').exists():
    print('Repo sudah ada, skip clone.')
else:
    print('Cloning repo...')
    !git lfs install 2>/dev/null || True
    !git clone https://github.com/milalestari/TUGAS-AKHIR.git . 2>/dev/null || True

print('Repo ready.')
print('dataset/split/' if (REPO / 'dataset' / 'split').exists() else 'MISSING: dataset/split/')
print('src/model_builder.py:', (REPO / 'src' / 'model_builder.py').exists())
print('src/train.py:', (REPO / 'src' / 'train.py').exists())

## Langkah 2 — Import Library

In [ ]:
import os, sys
from pathlib import Path
import random, json, time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from PIL import Image

import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU available:', bool(tf.config.list_physical_devices('GPU')))

# Versi GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print('  GPU:', gpu)

# Setup repo path
REPO = Path('/kaggle/input/tugas-akhir')  # adjust if using Kaggle dataset
if not REPO.exists():
    REPO = Path('.')

sys.path.insert(0, str(REPO / 'src'))
from src.model_builder import *
from src.train import train_two_stage, train_all_seeds

print('src/ imported OK')
print('SEEDS:', SEEDS)

## Langkah 3 — Verifikasi `data_pipeline`

In [ ]:
# Cek apakah data_pipeline bisa diimport dan run
try:
    from src.data_pipeline import build_tf_dataset, CLASS_NAMES
    print('data_pipeline imported OK')
    print('CLASS_NAMES:', CLASS_NAMES)
except ImportError as e:
    print('ERROR: data_pipeline tidak bisa diimport.')
    print('Pastikan src/data_pipeline.py ada di repo.')
    raise e

# Verifikasi folder
split_root = REPO / 'dataset' / 'split'
if not split_root.exists():
    print('WARNING: dataset/split/ tidak ditemukan.')
    print('Upload dataset ke Kaggle atau clone dari GitHub LFS.')
else:
    seed_folders = [f.name for f in split_root.iterdir() if f.is_dir()]
    print('Seeds found:', sorted(seed_folders))
    print('data_pipeline ready.')

## Langkah 4 — Load Datasets (Demo: seed=42)

In [ ]:
preprocess_resnet = tf.keras.applications.resnet50.preprocess_input

# Demo dengan seed=42
DEMO_SEED = 42

print(f'Loading datasets for seed={DEMO_SEED}...')
print('Train (augmented):')
ds_train_demo = build_tf_dataset(
    seed=DEMO_SEED,
    subset='train',
    preprocess_fn=preprocess_resnet,
    augment=True,
    batch_size=32,
)

print('Val (no augmentation):')
ds_val_demo = build_tf_dataset(
    seed=DEMO_SEED,
    subset='val',
    preprocess_fn=preprocess_resnet,
    augment=False,
    batch_size=32,
)

print('Test (no augmentation):')
ds_test_demo = build_tf_dataset(
    seed=DEMO_SEED,
    subset='test',
    preprocess_fn=preprocess_resnet,
    augment=False,
    batch_size=32,
)

# Hitung steps
steps_train = len(list(ds_train_demo))
steps_val   = len(list(ds_val_demo))
steps_test  = len(list(ds_test_demo))

print(f'Train: {steps_train} steps, Val: {steps_val} steps, Test: {steps_test} steps')
print('Class names:', CLASS_NAMES)

## Langkah 5 — Build ResNet50 Model (Stage 1: Freeze)

In [ ]:
# Bangun model Stage 1 (freeze)
print('Building ResNet50 (Stage 1: freeze)...')
model_demo, unfreeze_names, total_layers = build_model(
    architecture='resnet50',
    seed=DEMO_SEED,
    preprocess_fn=preprocess_resnet,
)

print_model_info(model_demo)
print()
print(f'Base model total layers : {total_layers}')
print(f'Layer di-unfreeze (Stage 2): {RESNET50_UNFREEZE} layer terakhir')
print(f'Unfreeze count: {RESNET50_UNFREEZE} layer terakhir')

## Langkah 6 — Stage 1: Feature Extraction (Freeze, 10 Epoch)

In [ ]:
print('=' * 60)
print('STAGE 1 — Feature Extraction (freeze, LR=1e-3, 10 epoch)')
print('=' * 60)

MODELS_DIR = REPO / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

stage1_path = str(MODELS_DIR / f'resnet50_seed{DEMO_SEED}_stage1.keras')
final_path  = str(MODELS_DIR / f'resnet50_seed{DEMO_SEED}.keras')

callbacks_s1 = build_callbacks(checkpoint_path=stage1_path)

print(f'Checkpoint: {stage1_path}')
print(f'Epochs: {EPOCHS_STAGE1}, LR: {STAGE1_LR}')
print()

t_start = time.time()
history_s1 = model_demo.fit(
    ds_train_demo,
    validation_data=ds_val_demo,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_s1,
    verbose=1,
)
t_stage1 = time.time() - t_start

print()
print(f'Stage 1 done in {t_stage1:.1f}s')
print(f'Best val_loss: {min(history_s1.history["val_loss"]):.4f}')
print(f'Best val_accuracy: {max(history_s1.history["val_accuracy"]):.4f}')

# Muat best weights
model_demo.load_weights(stage1_path)

## Langkah 7 — Stage 2: Fine-Tuning (Unfreeze, 10 Epoch)

In [ ]:
print('=' * 60)
print('STAGE 2 — Fine-Tuning (unfreeze top layers, LR=1e-4, 10 epoch)')
print('=' * 60)

# Unfreeze untuk fine-tuning
model_demo = unfreeze_for_fine_tune(model_demo)
print_model_info(model_demo)
print()
print(f'Unfreeze: {RESNET50_UNFREEZE} layer terakhir')
print(f'LR Stage 2: {STAGE2_LR}')
print(f'Epochs: {EPOCHS_STAGE2}')

callbacks_s2 = build_callbacks(checkpoint_path=final_path)

t_start = time.time()
history_s2 = model_demo.fit(
    ds_train_demo,
    validation_data=ds_val_demo,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_s2,
    verbose=1,
)
t_stage2 = time.time() - t_start

# Muat best weights final
model_demo.load_weights(final_path)

print()
print(f'Stage 2 done in {t_stage2:.1f}s')
print(f'Best val_loss: {min(history_s2.history["val_loss"]):.4f}')
print(f'Best val_accuracy: {max(history_s2.history["val_accuracy"]):.4f}')
print(f'Final checkpoint: {final_path}')

## Langkah 8 — Visualisasi Training History

In [ ]:
# Gabungkan history Stage 1 dan Stage 2
epochs_all = list(range(1, EPOCHS_STAGE1 + 1)) + list(range(EPOCHS_STAGE1 + 1, EPOCHS_STAGE1 + EPOCHS_STAGE2 + 1))

train_loss  = history_s1.history['loss'] + history_s2.history['loss']
val_loss    = history_s1.history['val_loss'] + history_s2.history['val_loss']
train_acc   = history_s1.history['accuracy'] + history_s2.history['accuracy']
val_acc     = history_s1.history['val_accuracy'] + history_s2.history['val_accuracy']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Stage divider
divider = EPOCHS_STAGE1

# Loss
axes[0].axvline(x=divider, color='red', linestyle='--', label='Stage 1->2')
axes[0].plot(epochs_all, train_loss, label='train_loss', marker='o', markersize=4)
axes[0].plot(epochs_all, val_loss, label='val_loss', marker='s', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].axvline(x=divider, color='red', linestyle='--', label='Stage 1->2')
axes[1].plot(epochs_all, train_acc, label='train_accuracy', marker='o', markersize=4)
axes[1].plot(epochs_all, val_acc, label='val_accuracy', marker='s', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'ResNet50 — seed={DEMO_SEED}', fontsize=12, y=1.02)
plt.tight_layout()

hist_fig_dir = REPO / 'results' / 'figures'
hist_fig_dir.mkdir(parents=True, exist_ok=True)
hist_path = hist_fig_dir / f'history_resnet50_seed{DEMO_SEED}.png'
plt.savefig(hist_path, dpi=150)
plt.show()
print('Saved:', hist_path)

## Langkah 9 — Evaluasi pada Test Set (seed=42)

In [ ]:
print('Evaluating on test set (seed=42)...')
print()

# Prediksi
y_true = []
y_pred = []
for batch_images, batch_labels in ds_test_demo:
    preds = model_demo.predict(batch_images, verbose=0)
    y_pred.extend(np.argmax(preds, axis=-1))
    y_true.extend(batch_labels.numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Accuracy
accuracy = np.mean(y_true == y_pred)
print(f'Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)')

# Classification report
from sklearn.metrics import classification_report, confusion_matrix
print()
print('Classification Report (macro avg):')
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES,
    digits=4,
))

## Langkah 10 — Confusion Matrix (Normalized)

In [ ]:
# Confusion matrix normalized
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt='.2f',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    ax=ax,
)
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'Confusion Matrix (Normalized) — ResNet50 seed={DEMO_SEED}')
plt.tight_layout()

cm_path = REPO / 'results' / 'figures' / f'cm_resnet50_seed{DEMO_SEED}.png'
plt.savefig(cm_path, dpi=150)
plt.show()
print('Saved:', cm_path)

## Langkah 11 — Simpan History & Metrik ke JSON

In [ ]:
# Simpan history
results_dir = REPO / 'results'
results_dir.mkdir(parents=True, exist_ok=True)

history_combined = {}
for k, v in history_s1.history.items():
    history_combined[f'stage1_{k}'] = v
for k, v in history_s2.history.items():
    history_combined[f'stage2_{k}'] = v

hist_json_path = results_dir / f'history_resnet50_seed{DEMO_SEED}.json'
with open(hist_json_path, 'w') as f:
    json.dump(history_combined, f, indent=2)
print('History JSON:', hist_json_path)

# Simpan metrik test
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

prec, rec, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average='macro'
)
acc = accuracy_score(y_true, y_pred)

metrics = {
    'architecture': 'resnet50',
    'seed': DEMO_SEED,
    'test_accuracy': float(acc),
    'test_precision_macro': float(prec),
    'test_recall_macro': float(rec),
    'test_f1_macro': float(f1),
    'best_val_loss_stage1': float(min(history_s1.history['val_loss'])),
    'best_val_loss_stage2': float(min(history_s2.history['val_loss'])),
    'time_stage1_s': float(t_stage1),
    'time_stage2_s': float(t_stage2),
    'epochs_stage1': EPOCHS_STAGE1,
    'epochs_stage2': EPOCHS_STAGE2,
}

metrics_path = results_dir / f'metrics_resnet50_seed{DEMO_SEED}.json'
with open(metrics_path, 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics JSON:', metrics_path)
print()
print('Test metrics:')
for k, v in metrics.items():
    if k not in ['architecture', 'seed', 'epochs_stage1', 'epochs_stage2']:
        print(f'  {k}: {v:.4f}')

## Langkah 12 — Full Pipeline: Semua 5 Seed (Sequential)

In [ ]:
# ================================================================
# FULL PIPELINE — 5 seed × ResNet50 × 2-stage training
# ================================================================
# JANGAN JALANKAN CELL INI SEKALIGUS DENGAN CELL 6-11
# Cell 6-11 adalah DEMO (seed=42). Ini pipeline untuk SEMUA seed.
# ================================================================

import gc

# Hapus model demo dari memory
del model_demo
del ds_train_demo, ds_val_demo, ds_test_demo
del history_s1, history_s2
gc.collect()
tf.keras.backend.clear_session()

print('=' * 60)
print('FULL PIPELINE — ResNet50 x 5 seeds')
print('=' * 60)

t_pipeline_start = time.time()
all_results = {}

for seed in SEEDS:
    print()
    print('=' * 60)
    print(f'SEED {seed} / {SEEDS}')
    print('=' * 60)

    t_seed_start = time.time()

    # Load datasets
    print('Loading datasets...')
    ds_train = build_tf_dataset(
        seed=seed, subset='train',
        preprocess_fn=preprocess_resnet,
        augment=True, batch_size=32,
    )
    ds_val = build_tf_dataset(
        seed=seed, subset='val',
        preprocess_fn=preprocess_resnet,
        augment=False, batch_size=32,
    )

    # Train
    ckpt_path = str(MODELS_DIR / f'resnet50_seed{seed}.keras')
    result = train_two_stage(
        architecture='resnet50',
        seed=seed,
        preprocess_fn=preprocess_resnet,
        ds_train=ds_train,
        ds_val=ds_val,
        checkpoint_path=ckpt_path,
        verbose=1,
    )
    all_results[seed] = result

    # Cleanup
    del result['model'], ds_train, ds_val
    gc.collect()
    tf.keras.backend.clear_session()

    t_seed = time.time() - t_seed_start
    print(f'Seed {seed} done in {t_seed:.1f}s')

t_total = time.time() - t_pipeline_start
print()
print('=' * 60)
print('ALL 5 SEEDS COMPLETE')
print(f'Total time: {t_total:.1f}s ({t_total/60:.1f} minutes)')
print('=' * 60)

## Langkah 13 — Simpan Semua Metrik (5 Seed)

In [ ]:
print('Saving all metrics...')
print()

all_metrics = []

for seed in SEEDS:
    mpath = results_dir / f'metrics_resnet50_seed{seed}.json'
    if mpath.exists():
        with open(mpath) as f:
            m = json.load(f)
        all_metrics.append(m)
        print(f'Seed {seed}: acc={m["test_accuracy"]:.4f} '
              f'f1={m["test_f1_macro"]:.4f}')

# Summary
if all_metrics:
    df = pd.DataFrame(all_metrics)
    print()
    print('=== SUMMARY ===')
    print(df[['seed','test_accuracy','test_precision_macro',
              'test_recall_macro','test_f1_macro']].to_string(index=False))
    print()
    print(f'Mean accuracy : {df["test_accuracy"].mean():.4f} +/- {df["test_accuracy"].std():.4f}')
    print(f'Mean F1 macro: {df["test_f1_macro"].mean():.4f} +/- {df["test_f1_macro"].std():.4f}')

    # Save summary
    summary_path = results_dir / 'metrics_resnet50_summary.csv'
    df.to_csv(summary_path, index=False)
    print()
    print('Summary saved:', summary_path)